# PART 1 — Setup, Utilities, Dataset Scanning

In [ ]:
import os
import re
import sys
import json
import time
import glob
import random
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import scipy
from scipy.signal import hilbert
from scipy.stats import skew, kurtosis

import sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, learning_curve, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier,
    ExtraTreesClassifier, VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    top_k_accuracy_score, RocCurveDisplay
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["figure.figsize"] = (10, 6)

SAMPLING_RATE_HZ = 250e6

BASE_DIR = Path.cwd()
DATA_DIR_CANDIDATES = [
    BASE_DIR / "Dataset 250 Msps",
    BASE_DIR / "Dataset_250_Msps" / "Dataset 250 Msps",
    BASE_DIR / "dataset" / "Dataset 250 Msps",
]
ZIP_CANDIDATES = [
    BASE_DIR / "Dataset_250_Msps.zip",
    BASE_DIR / "dataset" / "Dataset_250_Msps.zip",
]

def resolve_dataset_root():
    for cand in DATA_DIR_CANDIDATES:
        if cand.exists() and cand.is_dir():
            return cand
    for zp in ZIP_CANDIDATES:
        if zp.exists():
            extract_to = BASE_DIR / "dataset_extracted"
            extract_to.mkdir(exist_ok=True)
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(extract_to)
            for cand in extract_to.rglob("Dataset 250 Msps"):
                if cand.is_dir():
                    return cand
    for cand in BASE_DIR.rglob("Dataset 250 Msps"):
        if cand.is_dir():
            return cand
    raise FileNotFoundError(
        "Could not locate the 'Dataset 250 Msps' folder. "
        "Place the extracted folder or 'Dataset_250_Msps.zip' next to this notebook."
    )

DATA_ROOT = resolve_dataset_root()
print("Using dataset root:", DATA_ROOT)

OUTPUT_DIR = BASE_DIR / "outputs"
for sub in ["models", "plots", "reports", "predictions", "processed_data",
            "features", "checkpoints", "logs"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

BRAND_NAME_MAP = {
    "iphone": "Apple",
    "apple": "Apple",
    "lg": "LG",
    "samsung": "Samsung",
    "sony": "Sony",
    "xiaomi": "Xiaomi",
    "huawei": "Huawei",
}

EXPECTED_TRAIN_COUNT = 120
EXPECTED_TEST_COUNT = 30


def normalize_brand(raw_name):
    key = raw_name.strip().lower()
    return BRAND_NAME_MAP.get(key, raw_name.strip())


def load_signal(path):
    """Read a single-column BT signal .txt file into a 1-D numpy array."""
    values = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                values.append(float(line))
            except ValueError:
                continue
    return np.asarray(values, dtype=np.float64)


def normalize_signal(x):
    """Zero-mean, peak-normalized signal as described in the paper (Section 2.3)."""
    x = np.asarray(x, dtype=np.float64)
    x = x - np.mean(x)
    peak = np.max(np.abs(x))
    if peak == 0:
        return x
    return x / peak


def energy_envelope(x, win=25):
    """Smoothed normalized energy envelope used for transient detection."""
    energy = x ** 2
    win = max(3, min(win, len(x)))
    kernel = np.ones(win) / win
    env = np.convolve(energy, kernel, mode="same")
    m = env.max()
    if m > 0:
        env = env / m
    return env


def detect_transient(x, win=25, start_thresh=0.15, end_thresh=0.75):
    """
    Detect [noise | transient | steady-state] boundaries using the
    normalized energy envelope, following the transient-detection
    methodology described in the paper (Figures 4-6).
    Returns (transient_start_idx, transient_end_idx, envelope).
    """
    env = energy_envelope(x, win=win)
    above_start = np.where(env >= start_thresh)[0]
    above_end = np.where(env >= end_thresh)[0]
    n = len(x)
    if len(above_start) == 0 or len(above_end) == 0:
        return 0, n - 1, env
    t_start = int(above_start[0])
    candidates = above_end[above_end >= t_start]
    t_end = int(candidates[0]) if len(candidates) else int(above_end[0])
    if t_end <= t_start:
        t_end = min(t_start + 10, n - 1)
    return t_start, t_end, env


def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)


def save_fig(fig, name, subdir="plots"):
    path = OUTPUT_DIR / subdir / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


print("Environment ready. XGBoost available:", XGBOOST_AVAILABLE)

In [ ]:
records = []
for brand_dir in sorted(os.listdir(DATA_ROOT)):
    brand_path = DATA_ROOT / brand_dir
    if not brand_path.is_dir():
        continue
    brand = normalize_brand(brand_dir)
    for model_dir in sorted(os.listdir(brand_path)):
        model_path = brand_path / model_dir
        if not model_path.is_dir():
            continue
        model = model_dir.strip()
        for user_dir in sorted(os.listdir(model_path)):
            user_path = model_path / user_dir
            if not user_path.is_dir():
                continue
            is_test = user_dir.strip().lower().endswith("_test")
            user_id = re.sub(r"_test$", "", user_dir.strip(), flags=re.IGNORECASE)
            split = "test" if is_test else "train"
            device_id = f"{brand}_{model}_{user_id}"
            for fname in sorted(os.listdir(user_path)):
                if not fname.lower().endswith(".txt"):
                    continue
                records.append({
                    "brand": brand,
                    "model": model,
                    "user_id": user_id,
                    "device_id": device_id,
                    "split": split,
                    "filename": fname,
                    "path": str(user_path / fname),
                })

metadata_df = pd.DataFrame(records)
metadata_df.to_csv(OUTPUT_DIR / "processed_data" / "metadata.csv", index=False)

print("=" * 70)
print("DATASET SUMMARY (Dataset 250 Msps)")
print("=" * 70)
print("Total files found          :", len(metadata_df))
print("Unique brands               :", metadata_df["brand"].nunique())
print("Unique models                :", metadata_df["model"].nunique())
print("Unique devices (users)       :", metadata_df["device_id"].nunique())
print()
print("Files per split:")
print(metadata_df["split"].value_counts())
print()
print("Brand distribution (all files):")
print(metadata_df["brand"].value_counts())

In [ ]:
counts = metadata_df.groupby(["device_id", "split"]).size().unstack(fill_value=0)
if "train" not in counts.columns:
    counts["train"] = 0
if "test" not in counts.columns:
    counts["test"] = 0

bad_train = counts[counts["train"] != EXPECTED_TRAIN_COUNT]
bad_test = counts[counts["test"] != EXPECTED_TEST_COUNT]

if len(bad_train) > 0:
    warnings.warn(f"{len(bad_train)} device(s) do not have exactly "
                   f"{EXPECTED_TRAIN_COUNT} training files:\n{bad_train}")
else:
    print(f"OK: every device has exactly {EXPECTED_TRAIN_COUNT} training files.")

if len(bad_test) > 0:
    warnings.warn(f"{len(bad_test)} device(s) do not have exactly "
                   f"{EXPECTED_TEST_COUNT} test files:\n{bad_test}")
else:
    print(f"OK: every device has exactly {EXPECTED_TEST_COUNT} test files.")

display_counts = counts.copy()
display_counts.columns.name = None
display_counts

# PART 2 — Exploratory Data Analysis (Train Data Only)

In [ ]:
train_meta = metadata_df[metadata_df["split"] == "train"].reset_index(drop=True)
test_meta = metadata_df[metadata_df["split"] == "test"].reset_index(drop=True)
print("Train files:", len(train_meta), " | Test files (untouched during EDA):", len(test_meta))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = train_meta["brand"].value_counts().index
sns.countplot(data=train_meta, x="brand", order=order, palette="viridis", ax=ax)
ax.set_title("Brand Distribution (Train)")
ax.set_xlabel("Brand"); ax.set_ylabel("Count")
plt.xticks(rotation=30)
save_fig(fig, "eda_brand_distribution")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
order = train_meta["model"].value_counts().index
sns.countplot(data=train_meta, x="model", order=order, palette="mako", ax=ax)
ax.set_title("Model Distribution (Train)")
ax.set_xlabel("Model"); ax.set_ylabel("Count")
plt.xticks(rotation=75)
save_fig(fig, "eda_model_distribution")
plt.show()

In [ ]:
users_per_model = train_meta.groupby("model")["user_id"].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 6))
users_per_model.plot(kind="bar", color="teal", ax=ax)
ax.set_title("Number of Distinct Users (Devices) per Model")
ax.set_xlabel("Model"); ax.set_ylabel("Number of Users")
plt.xticks(rotation=75)
save_fig(fig, "eda_users_per_model")
plt.show()
users_per_model

In [ ]:
sample_n = min(400, len(train_meta))
sample_idx = train_meta.sample(sample_n, random_state=RANDOM_SEED).index
signal_lengths = []
for i in sample_idx:
    x = load_signal(train_meta.loc[i, "path"])
    signal_lengths.append(len(x))
train_meta.loc[sample_idx, "signal_length_sample"] = signal_lengths

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(signal_lengths, bins=30, kde=True, color="slateblue", ax=ax)
ax.set_title(f"Signal Length Histogram (sample of {sample_n} train files)")
ax.set_xlabel("Number of Samples"); ax.set_ylabel("Frequency")
save_fig(fig, "eda_signal_length_hist")
plt.show()
print("Signal length stats:", pd.Series(signal_lengths).describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
rand_rows = train_meta.sample(4, random_state=RANDOM_SEED).reset_index(drop=True)
for ax, (_, row) in zip(axes.flat, rand_rows.iterrows()):
    x = load_signal(row["path"])
    ax.plot(x, linewidth=0.6, color="darkorange")
    ax.set_title(f'{row["brand"]} {row["model"]} ({row["user_id"][:12]})', fontsize=11)
    ax.set_xlabel("Sample Index"); ax.set_ylabel("Amplitude")
fig.suptitle("Sample Raw Signals (Train)", fontsize=16)
plt.tight_layout()
save_fig(fig, "eda_sample_signals")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)
rand_rows = train_meta.sample(3, random_state=RANDOM_SEED + 1).reset_index(drop=True)
for ax, (_, row) in zip(axes, rand_rows.iterrows()):
    x = normalize_signal(load_signal(row["path"]))
    ax.plot(x, linewidth=0.6, color="crimson")
    ax.set_title(f'Waveform: {row["device_id"]}', fontsize=11)
    ax.set_ylabel("Normalized Amplitude")
axes[-1].set_xlabel("Sample Index")
fig.suptitle("Random Waveform Plots (Normalized, Train)", fontsize=16)
plt.tight_layout()
save_fig(fig, "eda_random_waveforms")
plt.show()

In [ ]:
missing_report = train_meta.isnull().sum()
print("Missing values per column (metadata):")
print(missing_report)

empty_files = []
for i in sample_idx[:200]:
    x = load_signal(train_meta.loc[i, "path"])
    if len(x) == 0 or np.any(np.isnan(x)):
        empty_files.append(train_meta.loc[i, "path"])
print(f"\nCorrupted/empty files detected in sampled subset: {len(empty_files)}")

In [ ]:
dup_count = train_meta.duplicated(subset=["brand", "model", "user_id", "filename"]).sum()
print("Duplicate metadata rows:", dup_count)

hash_map = {}
duplicate_signals = 0
for i in sample_idx[:200]:
    x = load_signal(train_meta.loc[i, "path"])
    h = hash(x.tobytes())
    if h in hash_map:
        duplicate_signals += 1
    else:
        hash_map[h] = train_meta.loc[i, "path"]
print("Duplicate raw signals detected in sampled subset:", duplicate_signals)

In [ ]:
amp_stats = []
for i in sample_idx:
    x = load_signal(train_meta.loc[i, "path"])
    amp_stats.append({
        "mean": np.mean(x), "std": np.std(x), "min": np.min(x),
        "max": np.max(x), "var": np.var(x)
    })
amp_stats_df = pd.DataFrame(amp_stats)
print("Amplitude statistics (sampled train signals):")
amp_stats_df.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(amp_stats_df["var"], bins=30, color="seagreen", kde=True, ax=ax)
ax.set_title("Signal Variance Distribution (Train Sample)")
ax.set_xlabel("Variance"); ax.set_ylabel("Frequency")
save_fig(fig, "eda_signal_variance")
plt.show()

In [ ]:
corr = amp_stats_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap (Amplitude Statistics)")
save_fig(fig, "eda_correlation_heatmap")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=amp_stats_df[["mean", "std", "min", "max"]], palette="Set2", ax=ax)
ax.set_title("Boxplots of Amplitude Statistics (Train Sample)")
save_fig(fig, "eda_boxplots")
plt.show()

q1 = amp_stats_df["std"].quantile(0.25)
q3 = amp_stats_df["std"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = amp_stats_df[(amp_stats_df["std"] < lower) | (amp_stats_df["std"] > upper)]
print(f"Outliers detected via IQR rule on std: {len(outliers)} / {len(amp_stats_df)}")

In [ ]:
print("Overall summary statistics (amplitude, sampled train signals):")
display_summary = amp_stats_df.describe().T
display_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(data=train_meta, x="brand", hue="split", palette="pastel", ax=ax)
ax.set_title("Class Balance Check — Brand (Train Only)")
plt.xticks(rotation=30)
save_fig(fig, "eda_class_balance_brand")
plt.show()

fig, ax = plt.subplots(figsize=(14, 5))
device_counts = train_meta["device_id"].value_counts()
device_counts.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Class Balance — Files per Device (Train)")
ax.set_xlabel("Device"); ax.set_ylabel("Count")
plt.xticks(rotation=90, fontsize=6)
save_fig(fig, "eda_class_balance_device")
plt.show()
print("Class balance (files per device) — min/max/mean:",
      device_counts.min(), device_counts.max(), device_counts.mean())

# PART 3 — Preprocessing (Train Data Only)

In [ ]:
def preprocess_signal(path, win=25, start_thresh=0.15, end_thresh=0.75):
    raw = load_signal(path)
    norm = normalize_signal(raw)
    t_start, t_end, env = detect_transient(norm, win=win,
                                            start_thresh=start_thresh,
                                            end_thresh=end_thresh)
    noise_region = norm[:t_start]
    transient_region = norm[t_start:t_end + 1]
    steady_region = norm[t_end + 1:]
    return {
        "raw": raw,
        "normalized": norm,
        "envelope": env,
        "transient_start": t_start,
        "transient_end": t_end,
        "noise_region": noise_region,
        "transient_region": transient_region,
        "steady_region": steady_region,
    }

In [ ]:
def plot_transient_regions(result, title, save_name=None):
    norm = result["normalized"]
    env = result["envelope"]
    t_start, t_end = result["transient_start"], result["transient_end"]

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(norm, linewidth=0.5, color="steelblue")
    axes[0].axvspan(0, t_start, color="lightgray", alpha=0.5, label="Noise")
    axes[0].axvspan(t_start, t_end, color="salmon", alpha=0.5, label="Transient")
    axes[0].axvspan(t_end, len(norm), color="lightgreen", alpha=0.4, label="Steady-State")
    axes[0].set_ylabel("Normalized Amplitude")
    axes[0].set_title(title)
    axes[0].legend(loc="upper right")

    axes[1].plot(env, linewidth=1.0, color="darkred")
    axes[1].axvline(t_start, color="black", linestyle="--", label="Transient Start")
    axes[1].axvline(t_end, color="black", linestyle=":", label="Transient End")
    axes[1].set_ylabel("Normalized Energy Envelope")
    axes[1].set_xlabel("Sample Index")
    axes[1].legend(loc="lower right")

    plt.tight_layout()
    if save_name:
        save_fig(fig, save_name)
    plt.show()

demo_rows = train_meta.sample(4, random_state=RANDOM_SEED + 7).reset_index(drop=True)
for i, row in demo_rows.iterrows():
    result = preprocess_signal(row["path"])
    plot_transient_regions(
        result,
        f'{row["brand"]} {row["model"]} — Noise / Transient / Steady-State',
        save_name=f"preprocessing_demo_{i}"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
colors = ["steelblue", "darkorange", "seagreen", "crimson"]
for i, row in demo_rows.iterrows():
    result = preprocess_signal(row["path"])
    ax.plot(result["envelope"], label=f'{row["brand"]} {row["model"]}',
            color=colors[i % len(colors)], linewidth=1.2)
ax.set_title("Comparison of Normalized Energy Envelopes")
ax.set_xlabel("Sample Index"); ax.set_ylabel("Normalized Energy")
ax.legend()
save_fig(fig, "envelope_comparison")
plt.show()

In [ ]:
t0 = time.time()
processed_records = []
for idx, row in train_meta.iterrows():
    result = preprocess_signal(row["path"])
    processed_records.append({
        "device_id": row["device_id"],
        "brand": row["brand"],
        "model": row["model"],
        "user_id": row["user_id"],
        "split": row["split"],
        "path": row["path"],
        "signal_length": len(result["raw"]),
        "transient_start": result["transient_start"],
        "transient_end": result["transient_end"],
        "transient_duration_samples": result["transient_end"] - result["transient_start"],
        "noise_len": len(result["noise_region"]),
        "transient_len": len(result["transient_region"]),
        "steady_len": len(result["steady_region"]),
    })
    if (idx + 1) % 500 == 0:
        print(f"Preprocessed {idx + 1}/{len(train_meta)} train signals...")

processed_train_df = pd.DataFrame(processed_records)
processed_train_df.to_csv(OUTPUT_DIR / "processed_data" / "processed_train_metadata.csv", index=False)
print(f"Preprocessing complete for {len(processed_train_df)} TRAIN signals in {time.time() - t0:.1f}s")
processed_train_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(processed_train_df["transient_duration_samples"], bins=40,
             color="purple", kde=True, ax=ax)
ax.set_title("Distribution of Detected Transient Duration (Train, samples)")
ax.set_xlabel("Transient Duration (samples)"); ax.set_ylabel("Frequency")
save_fig(fig, "transient_duration_distribution")
plt.show()
print(processed_train_df["transient_duration_samples"].describe())

# PART 4 — Feature Extraction (Hilbert / HOS / Envelope Statistics)

In [ ]:
def safe_entropy(x, bins=32):
    hist, _ = np.histogram(np.abs(x), bins=bins, density=True)
    hist = hist[hist > 0]
    if hist.sum() == 0:
        return 0.0
    p = hist / hist.sum()
    return float(-np.sum(p * np.log2(p)))


def stats_block(feats, name, arr):
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if len(arr) < 2:
        arr = np.array([0.0, 0.0])
    feats[f"{name}_mean"] = float(np.mean(arr))
    feats[f"{name}_var"] = float(np.var(arr))
    feats[f"{name}_skew"] = float(skew(arr))
    feats[f"{name}_kurt"] = float(kurtosis(arr))


def extract_feature_vector(path, fs=SAMPLING_RATE_HZ, win=25,
                            start_thresh=0.15, end_thresh=0.75):
    """
    Extract a single feature vector from one BT signal file, following
    the instantaneous-characteristics / higher-order-statistics (HOS)
    approach described in the paper (Section 3.1):
      - Hilbert Transform -> instantaneous amplitude, phase, frequency
      - HOS: mean, variance, skewness, kurtosis of each
      - Energy / entropy / RMS / peak / crest / shape / impulse factors
      - Transient duration, zero-crossing rate, envelope statistics
    """
    raw = load_signal(path)
    norm = normalize_signal(raw)
    t_start, t_end, env = detect_transient(norm, win=win,
                                            start_thresh=start_thresh,
                                            end_thresh=end_thresh)

    analytic_signal = hilbert(norm)
    inst_amplitude = np.abs(analytic_signal)
    inst_phase = np.unwrap(np.angle(analytic_signal))
    inst_frequency = np.diff(inst_phase) / (2.0 * np.pi) * fs

    feats = {}
    stats_block(feats, "inst_amp", inst_amplitude)
    stats_block(feats, "inst_phase", inst_phase)
    stats_block(feats, "inst_freq", inst_frequency)

    feats["entropy"] = safe_entropy(norm)
    feats["energy"] = float(np.sum(norm ** 2))
    feats["rms"] = float(np.sqrt(np.mean(norm ** 2)))

    peak = float(np.max(np.abs(norm)))
    mean_abs = float(np.mean(np.abs(norm))) + 1e-12
    feats["peak"] = peak
    feats["crest_factor"] = peak / (feats["rms"] + 1e-12)
    feats["shape_factor"] = feats["rms"] / mean_abs
    feats["impulse_factor"] = peak / mean_abs

    feats["transient_duration"] = (t_end - t_start) / fs
    feats["transient_duration_samples"] = float(t_end - t_start)

    zero_crossings = np.sum(np.diff(np.sign(norm)) != 0)
    feats["zero_crossing_rate"] = float(zero_crossings) / len(norm)

    feats["envelope_mean"] = float(np.mean(env))
    feats["envelope_var"] = float(np.var(env))
    feats["envelope_max"] = float(np.max(env))
    feats["envelope_skew"] = float(skew(env))
    feats["envelope_kurt"] = float(kurtosis(env))

    transient_region = norm[t_start:t_end + 1] if t_end > t_start else norm
    stats_block(feats, "transient_amp", transient_region)

    return feats


sample_feats = extract_feature_vector(train_meta.iloc[0]["path"])
print("Number of extracted features per signal:", len(sample_feats))
print(list(sample_feats.keys()))

In [ ]:
def build_feature_dataframe(meta_df, tag):
    t0 = time.time()
    rows = []
    for idx, row in meta_df.iterrows():
        feats = extract_feature_vector(row["path"])
        feats.update({
            "device_id": row["device_id"],
            "brand": row["brand"],
            "model": row["model"],
            "user_id": row["user_id"],
            "path": row["path"],
        })
        rows.append(feats)
        if (idx + 1) % 500 == 0:
            print(f"[{tag}] Extracted features for {idx + 1}/{len(meta_df)} signals...")
    fdf = pd.DataFrame(rows)
    print(f"[{tag}] Feature extraction complete: {fdf.shape} in {time.time() - t0:.1f}s")
    return fdf


train_features_df = build_feature_dataframe(train_meta, "TRAIN")
train_features_df.to_csv(OUTPUT_DIR / "features" / "train_features.csv", index=False)
train_features_df.head()

In [ ]:
FEATURE_COLUMNS = [c for c in train_features_df.columns
                    if c not in ("device_id", "brand", "model", "user_id", "path")]
with open(OUTPUT_DIR / "features" / "feature_columns.json", "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print(f"Total feature columns: {len(FEATURE_COLUMNS)}")
train_features_df[FEATURE_COLUMNS].describe().T

# PART 5 — Machine Learning (Training on TRAIN Features Only)

In [ ]:
X_train_raw = train_features_df[FEATURE_COLUMNS].values
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_features_df["device_id"].values)
CLASS_NAMES = label_encoder.classes_

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)

joblib.dump(scaler, OUTPUT_DIR / "models" / "scaler.pkl")
joblib.dump(label_encoder, OUTPUT_DIR / "models" / "label_encoder.pkl")

print("X_train shape:", X_train.shape)
print("Number of classes (devices/users):", len(CLASS_NAMES))

In [ ]:
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

model_defs = {}

model_defs["SVM_Poly2"] = SVC(kernel="poly", degree=2, C=1.0, coef0=1.0,
                               probability=True, random_state=RANDOM_SEED)

model_defs["RandomForest"] = RandomForestClassifier(
    n_estimators=300, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1)

model_defs["GradientBoosting"] = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.1, max_depth=3, random_state=RANDOM_SEED)

model_defs["AdaBoost"] = AdaBoostClassifier(
    n_estimators=200, learning_rate=0.5, random_state=RANDOM_SEED)

model_defs["ExtraTrees"] = ExtraTreesClassifier(
    n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1)

if XGBOOST_AVAILABLE:
    model_defs["XGBoost"] = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9, eval_metric="mlogloss",
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0)

print("Base models defined:", list(model_defs.keys()))

In [ ]:
PARAM_GRIDS = {
    "SVM_Poly2": {"C": [0.5, 1.0, 5.0], "coef0": [0.0, 1.0]},
    "RandomForest": {"n_estimators": [200, 300], "max_depth": [None, 20]},
}

tuned_models = {}
for name, model in model_defs.items():
    if name in PARAM_GRIDS:
        print(f"Tuning {name} with GridSearchCV (5-fold)...")
        grid = GridSearchCV(model, PARAM_GRIDS[name], cv=cv_splitter,
                             scoring="accuracy", n_jobs=-1)
        grid.fit(X_train, y_train)
        tuned_models[name] = grid.best_estimator_
        print(f"  Best params for {name}: {grid.best_params_}  "
              f"(CV accuracy={grid.best_score_:.4f})")
    else:
        tuned_models[name] = model

print("Tuned/base models ready:", list(tuned_models.keys()))

In [ ]:
estimators_for_ensemble = [(name, mdl) for name, mdl in tuned_models.items()
                            if name in ("SVM_Poly2", "RandomForest", "ExtraTrees")]

voting_clf = VotingClassifier(estimators=estimators_for_ensemble, voting="soft", n_jobs=-1)
tuned_models["VotingClassifier"] = voting_clf

stacking_clf = StackingClassifier(
    estimators=estimators_for_ensemble,
    final_estimator=LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
    cv=3, n_jobs=-1
)
tuned_models["StackingClassifier"] = stacking_clf

print("Final model roster:", list(tuned_models.keys()))

In [ ]:
cv_results_summary = []
trained_models = {}

for name, model in tuned_models.items():
    t0 = time.time()
    print(f"Training + 5-fold CV: {name} ...")
    cv_scores = cross_validate(
        model, X_train, y_train, cv=cv_splitter,
        scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"],
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    train_time = time.time() - t0
    trained_models[name] = model

    cv_results_summary.append({
        "model": name,
        "cv_accuracy_mean": cv_scores["test_accuracy"].mean(),
        "cv_accuracy_std": cv_scores["test_accuracy"].std(),
        "cv_precision_macro": cv_scores["test_precision_macro"].mean(),
        "cv_recall_macro": cv_scores["test_recall_macro"].mean(),
        "cv_f1_macro": cv_scores["test_f1_macro"].mean(),
        "train_time_sec": train_time,
    })
    joblib.dump(model, OUTPUT_DIR / "models" / f"{name}.pkl")
    print(f"  {name}: CV accuracy = {cv_scores['test_accuracy'].mean():.4f} "
          f"(+/- {cv_scores['test_accuracy'].std():.4f}), trained in {train_time:.1f}s")

cv_results_df = pd.DataFrame(cv_results_summary).sort_values("cv_accuracy_mean", ascending=False)
cv_results_df.to_csv(OUTPUT_DIR / "reports" / "cross_validation_results.csv", index=False)
cv_results_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=cv_results_df, x="model", y="cv_accuracy_mean", palette="viridis", ax=ax)
ax.set_title("5-Fold Cross-Validation Accuracy by Model (Train Data)")
ax.set_ylabel("CV Accuracy"); ax.set_xlabel("Model")
plt.xticks(rotation=30)
save_fig(fig, "cv_accuracy_by_model")
plt.show()

In [ ]:
best_train_model_name = cv_results_df.iloc[0]["model"]
best_train_model = trained_models[best_train_model_name]
print("Best model on CV (train data):", best_train_model_name)

train_preds = best_train_model.predict(X_train)
print("\nTraining classification report (best model, in-sample):")
print(classification_report(y_train, train_preds, target_names=CLASS_NAMES, zero_division=0))

cm_train = confusion_matrix(y_train, train_preds)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_train, cmap="Blues", ax=ax, cbar=True,
            xticklabels=False, yticklabels=False)
ax.set_title(f"Training Confusion Matrix — {best_train_model_name}")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
save_fig(fig, "train_confusion_matrix_best_model")
plt.show()

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    trained_models["RandomForest"], X_train, y_train, cv=3,
    train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1,
    scoring="accuracy", random_state=RANDOM_SEED
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Training score")
ax.plot(train_sizes, val_scores.mean(axis=1), "o-", label="Validation score")
ax.fill_between(train_sizes,
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15)
ax.fill_between(train_sizes,
                 val_scores.mean(axis=1) - val_scores.std(axis=1),
                 val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15)
ax.set_title("Learning Curve — RandomForest")
ax.set_xlabel("Training Examples"); ax.set_ylabel("Accuracy")
ax.legend()
save_fig(fig, "learning_curve_randomforest")
plt.show()

In [ ]:
if hasattr(trained_models["RandomForest"], "feature_importances_"):
    importances = pd.Series(trained_models["RandomForest"].feature_importances_,
                             index=FEATURE_COLUMNS).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 8))
    importances.head(20).plot(kind="barh", ax=ax, color="darkcyan")
    ax.invert_yaxis()
    ax.set_title("Feature Importance — RandomForest (Top 20)")
    ax.set_xlabel("Importance")
    save_fig(fig, "feature_importance_randomforest")
    plt.show()
    importances.to_csv(OUTPUT_DIR / "reports" / "feature_importance_randomforest.csv")

In [ ]:
for name, model in trained_models.items():
    joblib.dump(model, OUTPUT_DIR / "models" / f"{name}.pkl")
joblib.dump(FEATURE_COLUMNS, OUTPUT_DIR / "models" / "feature_columns.pkl")
print("All trained models saved to:", OUTPUT_DIR / "models")
print(sorted(os.listdir(OUTPUT_DIR / "models")))

# PART 6 — Testing on Held-Out `*_test` Folders

In [ ]:
test_features_df = build_feature_dataframe(test_meta, "TEST")
test_features_df.to_csv(OUTPUT_DIR / "features" / "test_features.csv", index=False)
test_features_df.head()

In [ ]:
scaler = joblib.load(OUTPUT_DIR / "models" / "scaler.pkl")
label_encoder = joblib.load(OUTPUT_DIR / "models" / "label_encoder.pkl")
FEATURE_COLUMNS = joblib.load(OUTPUT_DIR / "models" / "feature_columns.pkl")

X_test_raw = test_features_df[FEATURE_COLUMNS].values
X_test = scaler.transform(X_test_raw)          # transform ONLY, never fit
y_test = label_encoder.transform(test_features_df["device_id"].values)

print("X_test shape:", X_test.shape)

In [ ]:
model_files = sorted([f for f in os.listdir(OUTPUT_DIR / "models") if f.endswith(".pkl")
                       and f not in ("scaler.pkl", "label_encoder.pkl", "feature_columns.pkl")])
loaded_models = {f[:-4]: joblib.load(OUTPUT_DIR / "models" / f) for f in model_files}
print("Loaded models for testing:", list(loaded_models.keys()))

In [ ]:
test_results = []
all_predictions = {}
all_probabilities = {}

for name, model in loaded_models.items():
    t0 = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time() - t0
    all_predictions[name] = y_pred

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    roc_auc = np.nan
    top3_acc = np.nan
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
        all_probabilities[name] = y_proba
        try:
            roc_auc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
        except Exception:
            roc_auc = np.nan
        try:
            top3_acc = top_k_accuracy_score(y_test, y_proba, k=3,
                                             labels=np.arange(len(label_encoder.classes_)))
        except Exception:
            top3_acc = np.nan

    test_results.append({
        "model": name, "test_accuracy": acc, "test_precision_macro": prec,
        "test_recall_macro": rec, "test_f1_macro": f1, "test_roc_auc_macro": roc_auc,
        "top_3_accuracy": top3_acc, "inference_time_sec": inference_time,
    })
    print(f"{name:20s} | Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | "
          f"F1={f1:.4f} | ROC-AUC={roc_auc:.4f} | Top3={top3_acc:.4f}")

test_results_df = pd.DataFrame(test_results).sort_values("test_accuracy", ascending=False)
test_results_df.to_csv(OUTPUT_DIR / "reports" / "test_results.csv", index=False)
test_results_df

In [ ]:
best_test_model_name = test_results_df.iloc[0]["model"]
best_test_model = loaded_models[best_test_model_name]
y_pred_best = all_predictions[best_test_model_name]

print(f"Best model on held-out TEST data: {best_test_model_name}")
print("\nClassification report (TEST):")
print(classification_report(y_test, y_pred_best, target_names=label_encoder.classes_, zero_division=0))

In [ ]:
cm_test = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_test, cmap="Blues", ax=ax, cbar=True, xticklabels=False, yticklabels=False)
ax.set_title(f"Test Confusion Matrix — {best_test_model_name}")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
save_fig(fig, "test_confusion_matrix_best_model")
plt.show()

In [ ]:
eval_df = test_features_df[["device_id", "brand", "model", "user_id"]].copy()
eval_df["y_true"] = y_test
eval_df["y_pred"] = y_pred_best
eval_df["true_device"] = label_encoder.inverse_transform(eval_df["y_true"])
eval_df["pred_device"] = label_encoder.inverse_transform(eval_df["y_pred"])
eval_df["correct"] = eval_df["y_true"] == eval_df["y_pred"]

per_user_acc = eval_df.groupby("true_device")["correct"].mean().sort_values()
per_model_acc = eval_df.groupby("model")["correct"].mean().sort_values()
per_brand_acc = eval_df.groupby("brand")["correct"].mean().sort_values()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
per_brand_acc.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Per-Brand Accuracy (Test)")
per_model_acc.plot(kind="barh", ax=axes[1], color="darkorange")
axes[1].set_title("Per-Model Accuracy (Test)")
per_user_acc.plot(kind="barh", ax=axes[2], color="seagreen", fontsize=6)
axes[2].set_title("Per-User (Device) Accuracy (Test)")
plt.tight_layout()
save_fig(fig, "per_group_accuracy")
plt.show()

per_brand_acc.to_csv(OUTPUT_DIR / "reports" / "per_brand_accuracy.csv")
per_model_acc.to_csv(OUTPUT_DIR / "reports" / "per_model_accuracy.csv")
per_user_acc.to_csv(OUTPUT_DIR / "reports" / "per_user_accuracy.csv")

In [ ]:
misclassified = eval_df[~eval_df["correct"]].copy()
misclassified.to_csv(OUTPUT_DIR / "predictions" / "misclassified_samples.csv", index=False)
print(f"Total misclassified test samples: {len(misclassified)} / {len(eval_df)}")
misclassified.head(20)

In [ ]:
if best_test_model_name in all_probabilities:
    proba = all_probabilities[best_test_model_name]
    prob_df = pd.DataFrame(proba, columns=label_encoder.classes_)
    prob_df.insert(0, "true_device", eval_df["true_device"].values)
    prob_df.insert(1, "pred_device", eval_df["pred_device"].values)
    prob_df.to_csv(OUTPUT_DIR / "predictions" / "prediction_probabilities.csv", index=False)

    top3_preds = np.argsort(-proba, axis=1)[:, :3]
    top3_correct = np.array([y_test[i] in top3_preds[i] for i in range(len(y_test))])
    print(f"Top-3 accuracy ({best_test_model_name}): {top3_correct.mean():.4f}")
prob_df.head() if best_test_model_name in all_probabilities else None

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
for name in ["SVM_Poly2", "RandomForest", best_test_model_name]:
    if name not in all_probabilities:
        continue
    proba = all_probabilities[name]
    y_test_bin = pd.get_dummies(y_test).values
    try:
        fpr_tpr_auc = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=0.5)
        RocCurveDisplay.from_predictions(
            (y_test == y_test[0]).astype(int),
            proba[:, 0], name=f"{name} (class 0 vs rest)", ax=ax
        )
    except Exception:
        pass
ax.set_title("Example ROC Curves (one-vs-rest, first class)")
save_fig(fig, "roc_curve_example")
plt.show()

# PART 7 — Final Evaluation, Model Comparison, and Artifact Export

In [ ]:
import pickle

model_sizes = {}
for name in loaded_models:
    p = OUTPUT_DIR / "models" / f"{name}.pkl"
    model_sizes[name] = p.stat().st_size / (1024 * 1024)  # MB

memory_usage = {}
for name, model in loaded_models.items():
    memory_usage[name] = len(pickle.dumps(model)) / (1024 * 1024)  # MB

comparison_df = test_results_df.merge(cv_results_df, on="model", how="left")
comparison_df["model_size_mb"] = comparison_df["model"].map(model_sizes)
comparison_df["memory_usage_mb"] = comparison_df["model"].map(memory_usage)
comparison_df = comparison_df.sort_values("test_accuracy", ascending=False).reset_index(drop=True)
comparison_df.to_csv(OUTPUT_DIR / "reports" / "final_model_comparison.csv", index=False)
comparison_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
metrics_to_plot = [
    ("test_accuracy", "Accuracy Comparison"),
    ("test_precision_macro", "Precision Comparison"),
    ("test_recall_macro", "Recall Comparison"),
    ("test_f1_macro", "F1 Comparison"),
    ("test_roc_auc_macro", "ROC-AUC Comparison"),
    ("inference_time_sec", "Inference Time Comparison (s)"),
]
for ax, (col, title) in zip(axes.flat, metrics_to_plot):
    sns.barplot(data=comparison_df, x="model", y=col, ax=ax, palette="crest")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
save_fig(fig, "final_metric_comparisons")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=comparison_df, x="model_size_mb", y="test_accuracy",
                 hue="model", s=150, ax=ax, legend="brief")
ax.set_title("Model Size vs. Test Accuracy")
ax.set_xlabel("Model Size (MB)"); ax.set_ylabel("Test Accuracy")
save_fig(fig, "model_size_vs_accuracy")
plt.show()

In [ ]:
cv_table = comparison_df[["model", "cv_accuracy_mean", "cv_accuracy_std",
                           "cv_precision_macro", "cv_recall_macro", "cv_f1_macro"]]
print("Cross-validation summary table:")
cv_table

In [ ]:
best_overall = comparison_df.iloc[0]
print("=" * 70)
print("BEST MODEL SUMMARY")
print("=" * 70)
for k, v in best_overall.items():
    print(f"{k:25s}: {v}")

with open(OUTPUT_DIR / "reports" / "best_model_summary.json", "w") as f:
    json.dump(best_overall.to_dict(), f, indent=2, default=str)

BEST_MODEL_NAME = best_overall["model"]
BEST_MODEL = loaded_models[BEST_MODEL_NAME]
joblib.dump(BEST_MODEL, OUTPUT_DIR / "models" / "best_model.pkl")
print(f"\nAutomatically selected BEST MODEL: {BEST_MODEL_NAME}")
print("Saved to outputs/models/best_model.pkl")

In [ ]:
train_features_df.to_csv(OUTPUT_DIR / "features" / "train_features_final.csv", index=False)
test_features_df.to_csv(OUTPUT_DIR / "features" / "test_features_final.csv", index=False)
processed_train_df.to_csv(OUTPUT_DIR / "processed_data" / "processed_train_final.csv", index=False)
eval_df.to_csv(OUTPUT_DIR / "predictions" / "test_predictions_final.csv", index=False)

with open(OUTPUT_DIR / "reports" / "classification_report_best_model.txt", "w") as f:
    f.write(classification_report(y_test, y_pred_best,
                                   target_names=label_encoder.classes_, zero_division=0))

print("Final artifacts saved under:", OUTPUT_DIR)
for sub in sorted(os.listdir(OUTPUT_DIR)):
    sub_path = OUTPUT_DIR / sub
    if sub_path.is_dir():
        print(f"  {sub}/  ->  {len(os.listdir(sub_path))} file(s)")

In [ ]:
print("PIPELINE COMPLETE.")
print(f"Best model: {BEST_MODEL_NAME}")
print(f"Test accuracy: {best_overall['test_accuracy']:.4f}")
print(f"Test F1 (macro): {best_overall['test_f1_macro']:.4f}")
print("All models, scaler, label encoder, features, and reports saved to the outputs/ directory.")

# PART 8 — Streamlit Deployment

The Streamlit application is provided as a standalone file: `app.py`.
Run it with:

```bash
streamlit run app.py
```

`app.py` loads `outputs/models/best_model.pkl`, `outputs/models/scaler.pkl`, and `outputs/models/label_encoder.pkl` produced by Parts 5-7 of this notebook.